In [1]:
import csv
import stanza
import spacy
from supar import Parser
from conllu import parse_incr
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import re
import pandas as pd


/Users/frapadovani/Desktop/CHILDES-Parser/supar_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/frapadovani/Desktop/CHILDES-Parser/parser/supar/structs/fn.py:300: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float)
/Users/frapadovani/Desktop/CHILDES-Parser/parser/supar/structs/fn.py:308: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/Users/frapadovani/Desktop/CHILDES-Parser/parser/supar/structs/fn.py:320: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` i

In [2]:
turk_pipeline = stanza.Pipeline(lang='tr', processors='tokenize,pos,lemma,depparse', use_gpu=True)

2026-02-13 12:14:06 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


[2026-02-13 12:14:06 INFO] Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-02-13 12:14:06 INFO: Downloaded file to /Users/frapadovani/Desktop/extensive_reg/stanza/stanza/resources.json
2026-02-13 12:14:06 WARNING: Language tr package default expects mwt, which has been added


[2026-02-13 12:14:06 INFO] Downloaded file to /Users/frapadovani/Desktop/extensive_reg/stanza/stanza/resources.json
[2026-02-13 12:14:06 WARNING] Language tr package default expects mwt, which has been added


2026-02-13 12:14:17 INFO: Loading these models for language: tr (Turkish):
| Processor | Package       |
-----------------------------
| tokenize  | imst          |
| mwt       | imst          |
| pos       | imst_charlm   |
| lemma     | imst_nocharlm |
| depparse  | imst_charlm   |

2026-02-13 12:14:17 WARNING: GPU requested, but is not available!
2026-02-13 12:14:17 INFO: Using device: cpu
2026-02-13 12:14:17 INFO: Loading: tokenize


...

[2026-02-13 12:14:18 INFO] Loading: mwt
[2026-02-13 12:14:18 INFO] Loading: pos


2026-02-13 12:14:19 INFO: Loading: lemma
2026-02-13 12:14:19 INFO: Loading: depparse


[2026-02-13 12:14:19 INFO] Loading: lemma
[2026-02-13 12:14:19 INFO] Loading: depparse


2026-02-13 12:14:19 INFO: Done loading processors!


[2026-02-13 12:14:19 INFO] Done loading processors!


In [ ]:
sent = ""

### Parse the grammatical, ambiguous and ungrammatical sentences and save them 

In [2]:
def parse_and_save_conllu(data_file, output_folder, parsers):

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(data_file)

    categories = {
        "grammatical": df[df['is_grammatical'] == 1]['transcript_clean'].tolist(),
        "ambiguous": df[df['is_grammatical'] == 0]['transcript_clean'].tolist(),
        "ungrammatical": df[df['is_grammatical'] == -1]['transcript_clean'].tolist()
    }

    for parser_name, nlp in parsers.items():
        for cat_name, sentences in categories.items():
            conllu_lines = []

            for sent_id, sent_text in enumerate(sentences, start=1):

                # --- SuPar/Biaffine parsers ---
                if parser_name in ["Supar_CDS"]:
                    dataset = nlp.predict([sent_text], lang='en', prob=True, verbose=False)
                    sent = dataset[0]

                    for word, arc, rel in zip(sent.words, sent.arcs, sent.rels):
                        head = arc
                        conllu_lines.append(
                            f"{word}\t_\t_\t_\t_\t_\t{head}\t{rel}\t_\t_"
                        )

                # --- Stanza parser ---
                elif parser_name == "Stanza_off_the_shelf":
                    doc = nlp(sent_text)
                    for sent in doc.sentences:
                        for token in sent.tokens:
                            for word in token.words:
                                head = 0 if word.head == word.id else word.head
                                conllu_lines.append(
                                    f"{word.text}\t_\t{word.upos}\t{word.xpos}\t_\t_\t{head}\t{word.deprel}\t_\t_"
                                )

                # 🔹 ADD EOS MARKER AFTER EACH SENTENCE
                conllu_lines.append("##")

            out_file = output_folder / f"{cat_name}_{parser_name}.conllu"
            with open(out_file, 'w', encoding='utf-8') as f:
                for line in conllu_lines:
                    f.write(line + "\n")

            print(f"Saved output to {out_file}")


In [3]:
def parse_and_save_conllu(data_file, output_folder, parsers):

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(data_file)

    # Keep full rows, not just text
    categories = {
        "grammatical": df[df['is_grammatical'] == 1],
        "ambiguous": df[df['is_grammatical'] == 0],
        "ungrammatical": df[df['is_grammatical'] == -1]
    }

    for parser_name, nlp in parsers.items():
        for cat_name, subdf in categories.items():
            conllu_lines = []

            for sent_id, row in enumerate(subdf.itertuples(index=False), start=1):
                sent_text = row.transcript_clean
                labels = row.labels

                # ---- Sentence-level metadata ----
                conllu_lines.append(f"# sent_id = {sent_id}")
                conllu_lines.append(f"# text = {sent_text}")
                conllu_lines.append(f"# is_grammatical = {row.is_grammatical}")

                if cat_name == "ungrammatical":
                    conllu_lines.append(f"# labels = {labels}")

                # ---- SuPar / Biaffine parsers ----
                if parser_name in ["Supar_CDS"]:
                    dataset = nlp.predict([sent_text], lang='en', prob=True, verbose=False)
                    sent = dataset[0]

                    for i, (word, arc, rel) in enumerate(
                        zip(sent.words, sent.arcs, sent.rels), start=1
                    ):
                        conllu_lines.append(
                            f"{i}\t{word}\t_\t_\t_\t_\t{arc}\t{rel}\t_\t_"
                        )

                # ---- Stanza parser ----
                elif parser_name == "Stanza_off_the_shelf":
                    doc = nlp(sent_text)
                    for sent in doc.sentences:
                        for word in sent.words:
                            head = 0 if word.head == word.id else word.head
                            conllu_lines.append(
                                f"{word.id}\t{word.text}\t_\t{word.upos}\t{word.xpos}\t_\t"
                                f"{head}\t{word.deprel}\t_\t_"
                            )

                # ---- Sentence boundary ----
                conllu_lines.append("")  # standard CoNLL-U sentence separator

            out_file = output_folder / f"{cat_name}_{parser_name}.conllu"
            with open(out_file, 'w', encoding='utf-8') as f:
                f.write("\n".join(conllu_lines))

            print(f"Saved output to {out_file}")

In [4]:
# --- Example usage ---
parsers = {
    "Supar_CDS": Parser.load('/Users/frapadovani/Desktop/CHILDES-Parser/parser/parser_trained/biaffine_roberta_large_childes_10/brlc'),
    "Stanza_off_the_shelf": stanza.Pipeline(lang='en', processors='tokenize,pos,lemma,depparse', use_gpu=True)
}

data_file = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/manually_annotated_full.csv"
output_folder = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time"

parse_and_save_conllu(data_file, output_folder, parsers)

/Users/frapadovani/Desktop/CHILDES-Parser/parser/supar/parser.py:565: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(path, map_location='cpu')
Some weights

[2026-01-26 08:09:12 INFO] Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-01-26 08:09:12 INFO: Downloaded file to /Users/frapadovani/Desktop/extensive_reg/stanza/stanza/resources.json
2026-01-26 08:09:12 WARNING: Language en package default expects mwt, which has been added


[2026-01-26 08:09:12 INFO] Downloaded file to /Users/frapadovani/Desktop/extensive_reg/stanza/stanza/resources.json
[2026-01-26 08:09:12 WARNING] Language en package default expects mwt, which has been added


2026-01-26 08:09:13 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2026-01-26 08:09:13 WARNING: GPU requested, but is not available!
2026-01-26 08:09:13 INFO: Using device: cpu
2026-01-26 08:09:13 INFO: Loading: tokenize
2026-01-26 08:09:13 INFO: Loading: mwt
2026-01-26 08:09:13 INFO: Loading: pos


...

[2026-01-26 08:09:14 INFO] Loading: lemma


2026-01-26 08:09:15 INFO: Loading: depparse


[2026-01-26 08:09:15 INFO] Loading: depparse


2026-01-26 08:09:15 INFO: Done loading processors!


[2026-01-26 08:09:15 INFO] Done loading processors!
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/grammatical_Supar_CDS.conllu
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/ambiguous_Supar_CDS.conllu
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/ungrammatical_Supar_CDS.conllu
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/grammatical_Stanza_off_the_shelf.conllu
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/ambiguous_Stanza_off_the_shelf.conllu
Saved output to /Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time/ungrammatical_Stanza_off_the_shelf.conllu


### READ the parsed files and perform analysis

In [12]:
def read_eos_conllu(path):
    """
    Reads a conllu-like file where sentences are separated by '##'.
    Returns a list of dicts:
      {
        "tokens": [token lines],
        "metadata": {key: value}
      }
    """
    sentences = []
    current_tokens = []
    metadata = {}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            if line == "##":
                if current_tokens:
                    sentences.append({
                        "tokens": current_tokens,
                        "metadata": metadata
                    })
                current_tokens = []
                metadata = {}

            elif line.startswith("#"):
                if "=" in line:
                    key, val = line[1:].split("=", 1)
                    metadata[key.strip()] = val.strip()

            elif line:
                current_tokens.append(line)

    if current_tokens:
        sentences.append({
            "tokens": current_tokens,
            "metadata": metadata
        })

    return sentences


def read_eos_conllu_updated(path):
    """
    Reads a standard CoNLL-U file.
    Sentences are separated by blank lines.
    Returns a list of dicts:
      {
        "tokens": [token lines],
        "metadata": {key: value}
      }
    """
    sentences = []
    current_tokens = []
    metadata = {}

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            # Sentence boundary
            if not line:
                if current_tokens:
                    sentences.append({
                        "tokens": current_tokens,
                        "metadata": metadata
                    })
                current_tokens = []
                metadata = {}
                continue

            # Metadata
            if line.startswith("#"):
                if "=" in line:
                    key, val = line[1:].split("=", 1)
                    metadata[key.strip()] = val.strip()
                continue

            # Token line
            current_tokens.append(line)

    # Catch last sentence if file doesn't end with newline
    if current_tokens:
        sentences.append({
            "tokens": current_tokens,
            "metadata": metadata
        })

    return sentences



In [19]:
def dependency_structure(sentence):
    """
    Returns list of (HEAD, DEPREL) tuples
    """
    structure = []
    for line in sentence["tokens"]:
        fields = line.split("\t")
        if len(fields) >= 8:
            head = int(fields[6])
            deprel = fields[7]
            structure.append((head, deprel))
    return structure


def sentence_text(sentence):
    # FORM column = fields[1]
    return " ".join(line.split("\t")[1] for line in sentence["tokens"])

def annotation_as_string(sentence):
    return "\n".join(sentence["tokens"])


In [20]:
def compare_parsers_for_category(
    file_cds,
    file_stanza,
    output_csv,
    category_name
):
    sents_cds = read_eos_conllu_updated(file_cds)
    sents_sta = read_eos_conllu_updated(file_stanza)

    assert len(sents_cds) == len(sents_sta), \
        f"Sentence count mismatch in {category_name}"

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        header = [
            "Sentence",
            "Supar_CDS",
            "Stanza_off_the_shelf"
        ]

        if category_name == "ungrammatical":
            header.insert(1, "Label")

        writer.writerow(header)

        for sent_cds, sent_sta in zip(sents_cds, sents_sta):

            dep_cds = dependency_structure(sent_cds)
            dep_sta = dependency_structure(sent_sta)

            if not (dep_cds == dep_sta):

                row = [sentence_text(sent_cds)]

                if category_name == "ungrammatical":
                    label = sent_cds["metadata"].get("labels", "")
                    row.append(label)

                row.extend([
                    annotation_as_string(sent_cds),
                    annotation_as_string(sent_sta)
                ])

                writer.writerow(row)


In [ ]:
base = Path("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs_second_time")

compare_parsers_for_category(
    base / "ambiguous_Supar_CDS.conllu",
    base / "ambiguous_Stanza_off_the_shelf.conllu",
    "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/ambiguous_discordant.csv",
    "ambiguous"
)

compare_parsers_for_category(
    base / "grammatical_Supar_CDS.conllu",
    base / "grammatical_Stanza_off_the_shelf.conllu",
    "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/grammatical_discordant.csv",
    "grammatical"
)

compare_parsers_for_category(
    base / "ungrammatical_Supar_CDS.conllu",
    base / "ungrammatical_Stanza_off_the_shelf.conllu",
    "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/ungrammatical_discordant.csv",
    "ungrammatical"
)


In [23]:
import pandas as pd

# Load your CSV
csv_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/manually_annotated_full.csv"
df = pd.read_csv(csv_path)

# Count the number of sentences for each grammaticality label
counts = df['is_grammatical'].value_counts().sort_index()

print("Counts of sentences by grammaticality:")
print(counts)

# Optional: explicitly show 1, 0, -1 even if some are missing
for label in [-1, 0, 1]:
    print(f"{label}: {counts.get(label, 0)}")


Counts of sentences by grammaticality:
is_grammatical
-1.0    1333
 0.0     648
 1.0    2219
Name: count, dtype: int64
-1: 1333
0: 648
1: 2219


In [1]:
import pandas as pd 
ambiguous = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/ambiguous_discordant.csv")
ambiguous

,Sentence,Supar_CDS,Stanza_off_the_shelf
0,lift Purdie's tail .,1\tlift\t_\t_\t_\t_\t0\troot\t_\t_\n2\tPurdie'...,1\tlift\t_\tVERB\tVB\t_\t0\troot\t_\t_\n2\tPur...
1,you find Thomas's face .,1\tyou\t_\t_\t_\t_\t2\tnsubj\t_\t_\n2\tfind\t_...,1\tyou\t_\tPRON\tPRP\t_\t2\tnsubj\t_\t_\n2\tfi...
2,"Mummy , don't .","1\tMummy\t_\t_\t_\t_\t3\tvocative\t_\t_\n2\t,\...",1\tMummy\t_\tNOUN\tNN\t_\t3\tvocative\t_\t_\n2...
3,choo choo .,1\tchoo\t_\t_\t_\t_\t0\troot\t_\t_\n2\tchoo\t_...,1\tchoo\t_\tPROPN\tNNP\t_\t0\troot\t_\t_\n2\tc...
4,it's Mummy's house .,1\tit's\t_\t_\t_\t_\t3\tcop\t_\t_\n2\tMummy's\...,1\tit\t_\tPRON\tPRP\t_\t5\tnsubj\t_\t_\n2\t's\...
...,...,...,...
331,"a , a witch hat .","1\ta\t_\t_\t_\t_\t3\treparandum\t_\t_\n2\t,\t_...","1\ta\t_\tDET\tDT\t_\t5\treparandum\t_\t_\n2\t,..."
332,yeah that's how witches are page .,1\tyeah\t_\t_\t_\t_\t2\tdiscourse\t_\t_\n2\tth...,1\tyeah\t_\tINTJ\tUH\t_\t4\tdiscourse\t_\t_\n2...
333,it's on the they slide on their belly .,1\tit's\t_\t_\t_\t_\t3\tnsubj\t_\t_\n2\ton\t_\...,1\tit\t_\tPRON\tPRP\t_\t6\tnsubj:outer\t_\t_\n...
334,one two five six seven eight nine ten for elev...,1\tone\t_\t_\t_\t_\t0\troot\t_\t_\n2\ttwo\t_\t...,1\tone\t_\tNUM\tCD\t_\t6\tnummod\t_\t_\n2\ttwo...


In [2]:
grammatical = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/grammatical_discordant.csv")
grammatical

,Sentence,Supar_CDS,Stanza_off_the_shelf
0,oh yes .,1\toh\t_\t_\t_\t_\t0\troot\t_\t_\n2\tyes\t_\t_...,1\toh\t_\tINTJ\tUH\t_\t2\tdiscourse\t_\t_\n2\t...
1,"Mummy , you say help .","1\tMummy\t_\t_\t_\t_\t4\tvocative\t_\t_\n2\t,\...",1\tMummy\t_\tNOUN\tNN\t_\t4\tvocative\t_\t_\n2...
2,Wend Wendy's hat .,1\tWend\t_\t_\t_\t_\t3\treparandum\t_\t_\n2\tW...,1\tWend\t_\tVERB\tVB\t_\t0\troot\t_\t_\n2\tWen...
3,that's Wendy's hat there .,1\tthat's\t_\t_\t_\t_\t3\tcop\t_\t_\n2\tWendy'...,1\tthat\t_\tPRON\tDT\t_\t5\tnsubj\t_\t_\n2\t's...
4,I'm not a fireman .,1\tI'm\t_\t_\t_\t_\t4\tcop\t_\t_\n2\tnot\t_\t_...,1\tI\t_\tPRON\tPRP\t_\t5\tnsubj\t_\t_\n2\t'm\t...
...,...,...,...
1228,because it's big .,1\tbecause\t_\t_\t_\t_\t3\tmark\t_\t_\n2\tit's...,1\tbecause\t_\tSCONJ\tIN\t_\t4\tmark\t_\t_\n2\...
1229,"because , I just like it .","1\tbecause\t_\t_\t_\t_\t5\tmark\t_\t_\n2\t,\t_...",1\tbecause\t_\tSCONJ\tIN\t_\t5\tmark\t_\t_\n2\...
1230,with the with the corn .,1\twith\t_\t_\t_\t_\t2\tcase\t_\t_\n2\tthe\t_\...,1\twith\t_\tADP\tIN\t_\t2\tcase\t_\t_\n2\tthe\...
1231,then we put water in for for the big duck to s...,1\tthen\t_\t_\t_\t_\t3\tadvmod\t_\t_\n2\twe\t_...,1\tthen\t_\tADV\tRB\t_\t3\tadvmod\t_\t_\n2\twe...


In [26]:
ungrammatical = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/ungrammatical_discordant.csv")
ungrammatical

,Sentence,Label,Supar_CDS,Stanza_off_the_shelf
0,I think stick on here .,"subject, tense_aspect",1\tI\t_\t_\t_\t_\t2\tnsubj\t_\t_\n2\tthink\t_\...,1\tI\t_\tPRON\tPRP\t_\t2\tnsubj\t_\t_\n2\tthin...
1,wanna be Fireman Sam .,subject,1\twanna\t_\t_\t_\t_\t0\troot\t_\t_\n2\tbe\t_\...,1\twan\t_\tVERB\tVBP\t_\t0\troot\t_\t_\n2\tna\...
2,wanna talk now .,subject,1\twanna\t_\t_\t_\t_\t0\troot\t_\t_\n2\ttalk\t...,1\twan\t_\tVERB\tVBP\t_\t0\troot\t_\t_\n2\tna\...
3,"Purdie , that Fireman Sam .",verb,"1\tPurdie\t_\t_\t_\t_\t4\tvocative\t_\t_\n2\t,...",1\tPurdie\t_\tPROPN\tNNP\t_\t0\troot\t_\t_\n2\...
4,a ladder climb up .,other,1\ta\t_\t_\t_\t_\t2\tdet\t_\t_\n2\tladder\t_\t...,1\ta\t_\tDET\tDT\t_\t3\tdet\t_\t_\n2\tladder\t...
...,...,...,...,...
638,because that .,preposition,1\tbecause\t_\t_\t_\t_\t2\tmark\t_\t_\n2\tthat...,1\tbecause\t_\tADP\tIN\t_\t2\tcase\t_\t_\n2\tt...
639,"mommy but , Firstname really seen real leprech...",auxiliary,1\tmommy\t_\t_\t_\t_\t6\tvocative\t_\t_\n2\tbu...,1\tmommy\t_\tINTJ\tUH\t_\t6\tdiscourse\t_\t_\n...
640,yeah but .,"subject, verb",1\tyeah\t_\t_\t_\t_\t2\tdiscourse\t_\t_\n2\tbu...,1\tyeah\t_\tINTJ\tUH\t_\t0\troot\t_\t_\n2\tbut...
641,he swim he drinked it all up .,tense_aspect,1\the\t_\t_\t_\t_\t2\tnsubj\t_\t_\n2\tswim\t_\...,1\the\t_\tPRON\tPRP\t_\t2\tnsubj\t_\t_\n2\tswi...


### Count frequency of labels in the ungrammatical case

In [28]:
labels_expanded = (
    ungrammatical
        .assign(Label=ungrammatical["Label"].str.split(","))
        .explode("Label")
)

# Clean whitespace
labels_expanded["Label"] = labels_expanded["Label"].str.strip()

label_counts = labels_expanded["Label"].value_counts()

label_counts


Label
determiner             150
verb                   134
subject                120
other                  103
auxiliary               77
tense_aspect            66
object                  56
preposition             51
present_progressive     37
sv_agreement            36
possessive              21
plural                   8
Name: count, dtype: int64

In [32]:
gold = pd.read_csv(
    "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/manually_annotated_full.csv"
)

ungrammatical_gold = gold[gold["is_grammatical"] == -1.0]

labels_expanded = (
    ungrammatical_gold
        .dropna(subset=["labels"])      # remove rows with no label
        .assign(labels=lambda df: df["labels"].str.split(","))
        .explode("labels")
)

labels_expanded["labels"] = labels_expanded["labels"].str.strip()
gold_label_counts = labels_expanded["labels"].value_counts()
gold_label_counts

labels
determiner             342
subject                322
verb                   267
auxiliary              207
other                  162
tense_aspect           143
object                 116
present_progressive     78
preposition             75
sv_agreement            61
possessive              26
plural                  17
Name: count, dtype: int64

In [33]:
discordant_df = (
    label_counts
        .rename("Discordant")
        .rename_axis("Label")
        .reset_index()
)

gold_df = (
    gold_label_counts
        .rename("Total_ungrammatical")
        .rename_axis("Label")
        .reset_index()
)


In [34]:
disagreement_df = discordant_df.merge(
    gold_df,
    on="Label",
    how="left"
)

disagreement_df["Disagreement_rate"] = (
    disagreement_df["Discordant"] /
    disagreement_df["Total_ungrammatical"]
)


In [36]:
disagreement_df = disagreement_df.sort_values(
    by="Disagreement_rate",
    ascending=False
)
disagreement_df["Disagreement_rate_pct"] = (
    disagreement_df["Disagreement_rate"] * 100
).round(2)

disagreement_df = disagreement_df.sort_values(
    by="Disagreement_rate_pct",
    ascending=False
)

disagreement_df 

,Label,Discordant,Total_ungrammatical,Disagreement_rate,Disagreement_rate_pct
10,possessive,21,26,0.807692,80.77
7,preposition,51,75,0.680000,68.00
3,other,103,162,0.635802,63.58
9,sv_agreement,36,61,0.590164,59.02
1,verb,134,267,0.501873,50.19
6,object,56,116,0.482759,48.28
8,present_progressive,37,78,0.474359,47.44
11,plural,8,17,0.470588,47.06
5,tense_aspect,66,143,0.461538,46.15
0,determiner,150,342,0.438596,43.86


## Check accuracy for 3 classes of the CDL-specific parser vs off-the-shelf (WHERE THERE IS GOLDEN)

In [5]:
import pandas as pd
import numpy as np
from collections import defaultdict

def parse_conllu_string(conllu_str):
    """
    Parse a CoNLL-U format string into a list of token dictionaries.
    
    Returns:
    --------
    List of dicts with keys: id, form, head, deprel
    """
    if pd.isna(conllu_str) or conllu_str.strip() == '':
        return None
    
    tokens = []
    for line in conllu_str.strip().split('\n'):
        parts = line.split('\t')
        if len(parts) >= 8:
            try:
                token_id = int(parts[0])
                tokens.append({
                    'id': token_id,
                    'form': parts[1],
                    'head': int(parts[6]),
                    'deprel': parts[7]
                })
            except (ValueError, IndexError):
                continue
    
    return tokens if tokens else None

def compute_las_uas(gold_tokens, pred_tokens):
    """
    Compute Labeled Attachment Score (LAS) and Unlabeled Attachment Score (UAS).
    
    Parameters:
    -----------
    gold_tokens : list of dicts
        Gold standard parse
    pred_tokens : list of dicts
        Predicted parse
    
    Returns:
    --------
    dict with 'las', 'uas', 'total' keys
    """
    if not gold_tokens or not pred_tokens:
        return {'las': 0, 'uas': 0, 'total': 0}
    
    # Create mappings by token ID
    gold_map = {t['id']: t for t in gold_tokens}
    pred_map = {t['id']: t for t in pred_tokens}
    
    # Get common token IDs
    common_ids = set(gold_map.keys()) & set(pred_map.keys())
    
    if not common_ids:
        return {'las': 0, 'uas': 0, 'total': 0}
    
    las_correct = 0
    uas_correct = 0
    total = len(common_ids)
    
    for token_id in common_ids:
        gold_tok = gold_map[token_id]
        pred_tok = pred_map[token_id]
        
        # UAS: correct if head matches
        if gold_tok['head'] == pred_tok['head']:
            uas_correct += 1
            
            # LAS: correct if both head and deprel match
            if gold_tok['deprel'] == pred_tok['deprel']:
                las_correct += 1
    
    return {
        'las': las_correct,
        'uas': uas_correct,
        'total': total
    }

def evaluate_single_file(csv_file, file_label):
    """
    Evaluate both parsers on a single CSV file.
    
    Parameters:
    -----------
    csv_file : str
        Path to CSV file
    file_label : str
        Label for this file (e.g., 'grammatical', 'ungrammatical', 'ambiguous')
    
    Returns:
    --------
    dict with results for both parsers and sentence details
    """
    results = {
        'supar': {'las': 0, 'uas': 0, 'total': 0},
        'stanza': {'las': 0, 'uas': 0, 'total': 0}
    }
    
    sentences_with_gold = []
    
    print(f"\nProcessing {file_label}...")
    df = pd.read_csv(csv_file)
    
    # Filter rows that have gold annotation
    df_with_gold = df[df['Gold_annotation'].notna() & (df['Gold_annotation'].str.strip() != '')]
    
    print(f"  Found {len(df_with_gold)} sentences with gold annotations")
    
    for idx, row in df_with_gold.iterrows():
        sentence = row['Sentence']
        
        # Parse all three versions
        gold_tokens = parse_conllu_string(row['Gold_annotation'])
        supar_tokens = parse_conllu_string(row['Supar_CDS'])
        stanza_tokens = parse_conllu_string(row['Stanza_off_the_shelf'])
        
        if not gold_tokens:
            continue
        
        # Compute scores for Supar
        supar_scores = {'las': 0, 'uas': 0, 'total': 0}
        if supar_tokens:
            supar_scores = compute_las_uas(gold_tokens, supar_tokens)
            results['supar']['las'] += supar_scores['las']
            results['supar']['uas'] += supar_scores['uas']
            results['supar']['total'] += supar_scores['total']
        
        # Compute scores for Stanza
        stanza_scores = {'las': 0, 'uas': 0, 'total': 0}
        if stanza_tokens:
            stanza_scores = compute_las_uas(gold_tokens, stanza_tokens)
            results['stanza']['las'] += stanza_scores['las']
            results['stanza']['uas'] += stanza_scores['uas']
            results['stanza']['total'] += stanza_scores['total']
        
        sentences_with_gold.append({
            'sentence': sentence,
            'category': file_label,
            'gold_tokens': len(gold_tokens),
            'supar_las': supar_scores['las'],
            'supar_uas': supar_scores['uas'],
            'stanza_las': stanza_scores['las'],
            'stanza_uas': stanza_scores['uas'],
        })
    
    return results, sentences_with_gold

def print_results(results, label):
    """Print formatted results for a single category."""
    print(f"\n{'='*60}")
    print(f"{label.upper()}")
    print(f"{'='*60}")
    
    if results['supar']['total'] > 0:
        print("\nSuPar CDS Parser:")
        print(f"  Total tokens: {results['supar']['total']}")
        print(f"  UAS: {results['supar']['uas']}/{results['supar']['total']} = {results['supar']['uas']/results['supar']['total']*100:.2f}%")
        print(f"  LAS: {results['supar']['las']}/{results['supar']['total']} = {results['supar']['las']/results['supar']['total']*100:.2f}%")
    else:
        print("\nSuPar CDS Parser: No data")
    
    if results['stanza']['total'] > 0:
        print("\nStanza Off-the-shelf Parser:")
        print(f"  Total tokens: {results['stanza']['total']}")
        print(f"  UAS: {results['stanza']['uas']}/{results['stanza']['total']} = {results['stanza']['uas']/results['stanza']['total']*100:.2f}%")
        print(f"  LAS: {results['stanza']['las']}/{results['stanza']['total']} = {results['stanza']['las']/results['stanza']['total']*100:.2f}%")
    else:
        print("\nStanza Off-the-shelf Parser: No data")
    
    if results['supar']['total'] > 0 and results['stanza']['total'] > 0:
        print(f"\nComparison:")
        print(f"  UAS Difference (SuPar - Stanza): {(results['supar']['uas']/results['supar']['total'] - results['stanza']['uas']/results['stanza']['total'])*100:.2f} percentage points")
        print(f"  LAS Difference (SuPar - Stanza): {(results['supar']['las']/results['supar']['total'] - results['stanza']['las']/results['stanza']['total'])*100:.2f} percentage points")

# File paths with labels
files = [
    ("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/discordant_with_gold/grammatical_with_gold.csv", "grammatical"),
    ("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/discordant_with_gold/ungrammatical_with_gold.csv", "ungrammatical"),
    ("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/discordant_with_gold/ambiguous_with_gold.csv", "ambiguous")
]

# Store results for each category
all_results = {}
all_sentences = []

# Evaluate each file separately
for csv_file, label in files:
    results, sentences = evaluate_single_file(csv_file, label)
    all_results[label] = results
    all_sentences.extend(sentences)
    print_results(results, label)

# Compute overall statistics
print(f"\n{'='*60}")
print("OVERALL STATISTICS (ALL CATEGORIES COMBINED)")
print(f"{'='*60}")

overall_supar = {'las': 0, 'uas': 0, 'total': 0}
overall_stanza = {'las': 0, 'uas': 0, 'total': 0}

for label, results in all_results.items():
    overall_supar['las'] += results['supar']['las']
    overall_supar['uas'] += results['supar']['uas']
    overall_supar['total'] += results['supar']['total']
    
    overall_stanza['las'] += results['stanza']['las']
    overall_stanza['uas'] += results['stanza']['uas']
    overall_stanza['total'] += results['stanza']['total']

print("\nSuPar CDS Parser (Overall):")
print(f"  Total tokens: {overall_supar['total']}")
print(f"  UAS: {overall_supar['uas']}/{overall_supar['total']} = {overall_supar['uas']/overall_supar['total']*100:.2f}%")
print(f"  LAS: {overall_supar['las']}/{overall_supar['total']} = {overall_supar['las']/overall_supar['total']*100:.2f}%")

print("\nStanza Off-the-shelf Parser (Overall):")
print(f"  Total tokens: {overall_stanza['total']}")
print(f"  UAS: {overall_stanza['uas']}/{overall_stanza['total']} = {overall_stanza['uas']/overall_stanza['total']*100:.2f}%")
print(f"  LAS: {overall_stanza['las']}/{overall_stanza['total']} = {overall_stanza['las']/overall_stanza['total']*100:.2f}%")

print(f"\nOverall Comparison:")
print(f"  UAS Difference (SuPar - Stanza): {(overall_supar['uas']/overall_supar['total'] - overall_stanza['uas']/overall_stanza['total'])*100:.2f} percentage points")
print(f"  LAS Difference (SuPar - Stanza): {(overall_supar['las']/overall_supar['total'] - overall_stanza['las']/overall_stanza['total'])*100:.2f} percentage points")

# Save detailed results to CSV
details_df = pd.DataFrame(all_sentences)
output_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/evaluation_details_by_category.csv"
details_df.to_csv(output_path, index=False)
print(f"\n\nDetailed results saved to: {output_path}")
print(f"Total sentences with gold annotations: {len(all_sentences)}")

# Create summary table
summary_data = []
for label in ['grammatical', 'ungrammatical', 'ambiguous']:
    if label in all_results:
        r = all_results[label]
        summary_data.append({
            'Category': label,
            'Total_Tokens': r['supar']['total'],
            'SuPar_UAS': f"{r['supar']['uas']/r['supar']['total']*100:.2f}%" if r['supar']['total'] > 0 else "N/A",
            'SuPar_LAS': f"{r['supar']['las']/r['supar']['total']*100:.2f}%" if r['supar']['total'] > 0 else "N/A",
            'Stanza_UAS': f"{r['stanza']['uas']/r['stanza']['total']*100:.2f}%" if r['stanza']['total'] > 0 else "N/A",
            'Stanza_LAS': f"{r['stanza']['las']/r['stanza']['total']*100:.2f}%" if r['stanza']['total'] > 0 else "N/A",
        })

summary_df = pd.DataFrame(summary_data)
summary_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/evaluation_summary_by_category.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")

print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(summary_df.to_string(index=False))


Processing grammatical...
  Found 155 sentences with gold annotations

GRAMMATICAL

SuPar CDS Parser:
  Total tokens: 512
  UAS: 473/512 = 92.38%
  LAS: 447/512 = 87.30%

Stanza Off-the-shelf Parser:
  Total tokens: 512
  UAS: 247/512 = 48.24%
  LAS: 176/512 = 34.38%

Comparison:
  UAS Difference (SuPar - Stanza): 44.14 percentage points
  LAS Difference (SuPar - Stanza): 52.93 percentage points

Processing ungrammatical...
  Found 31 sentences with gold annotations

UNGRAMMATICAL

SuPar CDS Parser:
  Total tokens: 110
  UAS: 101/110 = 91.82%
  LAS: 88/110 = 80.00%

Stanza Off-the-shelf Parser:
  Total tokens: 110
  UAS: 64/110 = 58.18%
  LAS: 56/110 = 50.91%

Comparison:
  UAS Difference (SuPar - Stanza): 33.64 percentage points
  LAS Difference (SuPar - Stanza): 29.09 percentage points

Processing ambiguous...
  Found 63 sentences with gold annotations

AMBIGUOUS

SuPar CDS Parser:
  Total tokens: 227
  UAS: 224/227 = 98.68%
  LAS: 211/227 = 92.95%

Stanza Off-the-shelf Parser:
  To

## CHECK HOW MANY MANUALLY ANNOTATED DATA (for grammaticality) are included in the golden parses

In [70]:
from pathlib import Path
import pandas as pd
import re

def normalize_sentence(sent):
    sent = sent.lower()
    sent = sent.replace("’", "'")             # <-- fixed
    sent = re.sub(r"[^\w\s']", "", sent)      # remove punctuation except apostrophe
    sent = re.sub(r"\s+", " ", sent)
    return sent.strip()



def read_conllu_sentences(conllu_path):
    sentences = []

    with open(conllu_path, "r", encoding="utf-8") as f:
        block = []
        tokens = []

        for line in f:
            line = line.rstrip("\n")

            if line == "":
                if tokens:
                    sent_text = " ".join(tokens)
                    norm_text = normalize_sentence(sent_text)

                    sentences.append({
                        "text": sent_text,
                        "norm": norm_text,
                        "block": "\n".join(block) + "\n"
                    })

                block = []
                tokens = []
                continue

            block.append(line)

            if not line.startswith("#"):
                cols = line.split("\t")
                if "-" not in cols[0] and "." not in cols[0]:
                    tokens.append(cols[1])

    return sentences




manual_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/manually_annotated_full.csv"

manual_df = pd.read_csv(manual_path)

manual_df["norm"] = manual_df["transcript_clean"].astype(str).apply(normalize_sentence)

manual_set = set(manual_df["norm"])



train_path = "/Users/frapadovani/Desktop/CHILDES-Parser/UD_English-CHILDES/en_childes-ud-train.conllu"
dev_path   = "/Users/frapadovani/Desktop/CHILDES-Parser/UD_English-CHILDES/en_childes-ud-dev.conllu"
test_path  = "/Users/frapadovani/Desktop/CHILDES-Parser/UD_English-CHILDES/en_childes-ud-test.conllu"

train_sents = read_conllu_sentences(train_path)
dev_sents   = read_conllu_sentences(dev_path)
test_sents  = read_conllu_sentences(test_path)

all_gold = (
    [("train", s) for s in train_sents] +
    [("dev", s)   for s in dev_sents] +
    [("test", s)  for s in test_sents]
)


matched = []

for split, sent in all_gold:
    if sent["norm"] in manual_set:
        matched.append({
            "split": split,
            "text": sent["text"],
            "block": sent["block"]
        })


output_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/matched_manual_sentences.conllu"

with open(output_path, "w", encoding="utf-8") as f:
    for item in matched:
        f.write(f"# source_split = {item['split']}\n")
        f.write(item["block"])
        f.write("\n")


matched_df = pd.DataFrame(matched)

print("Total matched sentences:", len(matched_df))
print(matched_df["split"].value_counts())


Total matched sentences: 925
split
train    492
test     373
dev       60
Name: count, dtype: int64
